In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import h5py

In [3]:
import torch
from lerobot.policies.smolvla import SmolVLAPolicy

/media/skpro19/ssd/nano-vla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
file = f"datasets/libero_spatial/pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate_demo.hdf5"

In [5]:
model_id = "lerobot/smolvla_base"


In [48]:
base_policy = SmolVLAPolicy.from_pretrained(model_id)

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


Loading weights: 100%|█| 489/489 [00:00


Reducing the number of VLM layers to 16 ...


In [49]:
isinstance(base_policy, torch.nn.Module)

True

In [50]:
base_policy.name

'smolvla'

In [51]:
input_features = base_policy.config.input_features

In [52]:
type(input_features)

dict

In [53]:
input_features.keys()

dict_keys(['observation.state', 'observation.images.camera1', 'observation.images.camera2', 'observation.images.camera3'])

In [54]:
print(f"input_features => {input_features}")

input_features => {'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(6,)), 'observation.images.camera1': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 256, 256)), 'observation.images.camera2': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 256, 256)), 'observation.images.camera3': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 256, 256))}


In [55]:
for k,v in input_features.items():
    print(f"{k}=> {type(v)}")

observation.state=> <class 'lerobot.configs.types.PolicyFeature'>
observation.images.camera1=> <class 'lerobot.configs.types.PolicyFeature'>
observation.images.camera2=> <class 'lerobot.configs.types.PolicyFeature'>
observation.images.camera3=> <class 'lerobot.configs.types.PolicyFeature'>


### Analyzing observation.state

In [57]:
obs_state = input_features["observation.state"]

In [58]:
type(obs_state)

lerobot.configs.types.PolicyFeature

In [59]:
print(f"obs_state.type => {obs_state.type} obs_state.shape => {obs_state.shape}")

obs_state.type => FeatureType.STATE obs_state.shape => (6,)


In [60]:
# Analyzing observation.images.camera1

In [61]:
obs_img_cam1 = input_features["observation.images.camera1"]

In [62]:
type(obs_img_cam1)

lerobot.configs.types.PolicyFeature

In [63]:
print(f"obs_img_cam1.type => {obs_img_cam1.type} obs_img_cam1.shape => {obs_img_cam1.shape}")

obs_img_cam1.type => FeatureType.VISUAL obs_img_cam1.shape => (3, 256, 256)


In [64]:
obs_img_cam2 = input_features["observation.images.camera2"]
print(f"obs_img_cam2.type => {obs_img_cam2.type} obs_img_cam2.shape => {obs_img_cam2.shape}")

obs_img_cam2.type => FeatureType.VISUAL obs_img_cam2.shape => (3, 256, 256)


In [65]:
# output features

In [67]:
output_features = base_policy.config.output_features

In [68]:
type(output_features)

dict

In [69]:
output_features.keys()

dict_keys(['action'])

In [70]:
output_features["action"].shape

(6,)

In [71]:
output_features["action"].type

<FeatureType.ACTION: 'ACTION'>

In [35]:
# preprocessing pipeline

In [47]:
from lerobot.policies.factory import make_pre_post_processors

### rename processor

In [113]:
type(pre.steps)


list

In [114]:
len(pre.steps)

6

In [115]:
pre.steps[0]

RenameObservationsProcessorStep(rename_map={})

In [117]:
pre.steps[0].rename_map 

{'observation.ee_states': 'observation.state',
 'observation.images.agentview_rgb': 'observation.image',
 'observation.images.eye_in_hand_rgb': 'observation.image2'}

In [118]:
## build data dict 

In [123]:
path = (
    "../datasets/libero_spatial/"
    "pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate_demo.hdf5"
)

In [124]:
from pathlib import Path

In [126]:
file = h5py.File(path, "r")
obs = file["data/demo_0/obs"]

In [127]:
type(obs)

h5py._hl.group.Group

In [128]:
ee_pos = obs["ee_pos"]

In [129]:
type(ee_pos) 

h5py._hl.dataset.Dataset

In [131]:
len(ee_pos)

98

In [132]:
ee_pos.shape

(98, 3)

In [133]:
obs.keys()

<KeysViewHDF5 ['agentview_rgb', 'ee_ori', 'ee_pos', 'ee_states', 'eye_in_hand_rgb', 'gripper_states', 'joint_states']>

In [138]:
for i, key in enumerate(obs.keys()):
    print(f"{key}.shape => {obs[key].shape}")

agentview_rgb.shape => (98, 128, 128, 3)
ee_ori.shape => (98, 3)
ee_pos.shape => (98, 3)
ee_states.shape => (98, 6)
eye_in_hand_rgb.shape => (98, 128, 128, 3)
gripper_states.shape => (98, 2)
joint_states.shape => (98, 7)


In [152]:
import numpy as np

In [160]:
data = {
    "observation.joint_states" : obs["joint_states"][0],
    "observation.gripper_states" : obs["gripper_states"][0],
    "observation.cam_top" : obs["agentview_rgb"][0],
    "observation.cam_wrist" : obs["eye_in_hand_rgb"][0],
    "observation.joint_states" : obs["joint_states"][0],
    "observation.state" : np.concat([obs["joint_states"][0], obs["gripper_states"][0]], axis=0)
}
    

In [161]:
data.keys()

dict_keys(['observation.joint_states', 'observation.gripper_states', 'observation.cam_top', 'observation.cam_wrist', 'observation.state'])

In [166]:
data["observation.state"].shape

(9,)

In [167]:
data["observation.state"]

array([ 0.00851556, -0.1760291 , -0.01122552, -2.4517907 , -0.00993433,
        2.21408242,  0.79684344,  0.036221  , -0.03623695])

### exploring pre-processing pipeline

In [150]:
for index, step in enumerate(pre.steps):
    print(index, type(step).__name__)

0 RenameObservationsProcessorStep
1 AddBatchDimensionProcessorStep
2 NewLineTaskProcessorStep
3 TokenizerProcessorStep
4 DeviceProcessorStep
5 NormalizerProcessorStep


In [168]:
# Map LIBERO's observation keys to the names expected by the loaded base policy.
pre.steps[0].rename_map = {
    "observation.images.agentview_rgb": "observation.images.camera1",
    "observation.images.eye_in_hand_rgb": "observation.images.camera2",
}

assert set(pre.steps[0].rename_map.values()).issubset(base_policy.config.input_features)